# Netflix title type prediction

Classify each catalog entry as a Movie or TV Show using its description, genres, country, rating, and release year. This notebook covers data preparation, exploratory analysis, model selection, tuning, holdout evaluation, and model-specific feature importance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                             ConfusionMatrixDisplay, f1_score)
from sklearn.model_selection import train_test_split

from model import make_search, prepare_data

## Load, inspect, and clean

The dataset is committed as `netflix_titles.csv`. The target is `type`; missing target rows and unsupported labels are removed. Missing text is made empty, missing ratings are labeled `Unknown`, and release year is converted to numeric so malformed values can be imputed by the model pipeline.

In [ ]:
raw = pd.read_csv("netflix_titles.csv")
print(f"Rows and columns: {raw.shape}")
display(raw.head())
display(raw.isna().sum().sort_values(ascending=False).to_frame("missing"))

X, y = prepare_data(raw)
print(f"Rows after target cleaning: {len(X)}")
display(y.value_counts().rename_axis("type").to_frame("count"))
display(pd.crosstab(raw["type"], raw["rating"].fillna("Unknown"),
                    margins=True).tail(6))

## Exploratory analysis

Inspect class balance, missingness, release-year distribution, rating distribution, outliers, and numeric correlations. Release year is the only numeric input; correlating it with a binary encoding of `type` is descriptive, not a feature-selection rule. The identifier, title, cast, and duration columns are not model inputs, avoiding title-derived leakage and keeping the feature set interpretable.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
y.value_counts().plot.bar(ax=axes[0, 0], title="Titles by type", rot=0)
raw["release_year"].dropna().plot.hist(bins=30, ax=axes[0, 1],
                                      title="Release-year distribution")
raw.boxplot(column="release_year", by="type", ax=axes[1, 0])
axes[1, 0].set_title("Release year by type")
raw["rating"].fillna("Unknown").value_counts().head(10).plot.bar(
    ax=axes[1, 1], title="Most common ratings", rot=45)
plt.tight_layout()
plt.show()

year = raw["release_year"].dropna()
q1, q3 = year.quantile([0.25, 0.75])
iqr = q3 - q1
print("Release-year IQR outliers:", int(((year < q1 - 1.5 * iqr) | (year > q3 + 1.5 * iqr)).sum()))
corr_data = pd.DataFrame({"release_year": raw["release_year"],
                          "is_tv_show": (raw["type"] == "TV Show").astype(float)})
display(corr_data.corr().round(3))
display(raw.isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame())
display(pd.crosstab(raw["type"], raw["rating"].fillna("Unknown"), normalize="index").round(3))

## Model selection and parameter tuning

A stratified holdout set is reserved for final evaluation. Five-fold cross-validation compares a Logistic Regression baseline, LinearSVC, and a shallow decision tree while tuning regularization, tree depth, and leaf size. Macro F1 is the selection metric because the catalog classes are imbalanced; balanced accuracy is reported alongside it. Every imputer and vectorizer is inside the pipeline, so it is fit only on each training fold.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
search = make_search(cv=5)
search.fit(X_train, y_train)
print("Best CV macro F1:", round(search.best_score_, 4))
print("Selected settings:", search.best_params_)
cv_results = pd.DataFrame(search.cv_results_)
cv_results["model"] = cv_results["param_classifier"].map(lambda model: model.__class__.__name__)
model_summary = (cv_results.sort_values("mean_test_macro_f1", ascending=False)
                 .groupby("model", as_index=False).first()
                 [["model", "mean_test_macro_f1", "mean_test_balanced_accuracy", "params"]]
                 .rename(columns={"mean_test_macro_f1": "cv_macro_f1",
                                  "mean_test_balanced_accuracy": "cv_balanced_accuracy"})
                 .sort_values("cv_macro_f1", ascending=False))
display(model_summary)

## Holdout results and feature weights

The selected pipeline is evaluated once on held-out data. Linear models provide signed coefficients by class; the tree provides impurity-based feature importance. Both are model-specific associations, not causal effects.

In [ ]:
best_model = search.best_estimator_
predictions = best_model.predict(X_test)
print(classification_report(y_test, predictions, zero_division=0))
print("Holdout macro F1:", round(f1_score(y_test, predictions, average="macro"), 4))
print("Holdout balanced accuracy:", round(balanced_accuracy_score(y_test, predictions), 4))
ConfusionMatrixDisplay.from_predictions(y_test, predictions)
plt.tight_layout()
plt.show()

classifier = best_model.named_steps["classifier"]
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
if hasattr(classifier, "coef_"):
    weights = classifier.coef_[0]
    classes = classifier.classes_
    for label, indices in ((classes[0], weights.argsort()[:15]),
                           (classes[1], weights.argsort()[-15:][::-1])):
        print(f"\nFeatures associated with {label}:")
        display(pd.DataFrame({"feature": feature_names[indices], "signed_coefficient": weights[indices]}))
else:
    importances = classifier.feature_importances_
    indices = importances.argsort()[-15:][::-1]
    display(pd.DataFrame({"feature": feature_names[indices], "importance": importances[indices]}))